In [2]:
from pathlib import Path
import sys
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path('/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR')
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Experimentation with Transformer Time-Series Models

This notebook follows the same experimental idea as `experiment_pyod_models.ipynb`, but adapted to the time-series stack in RADAR.

It benchmarks three Transformer-based models available in `RADAR.time_series.algorithms.transformers` on two UCI datasets used in the existing examples: `ai4i_2020_predictive_maintenance_dataset` and `metro_interstate_traffic_volume`.

## Import Required Libraries

Imports for dataset loading, preprocessing, window generation, model training, metrics, and result persistence.

In [3]:
import importlib
import time
from pathlib import Path

import numpy as np
import pandas as pd

from RADAR.time_series.algorithms import transformers
from RADAR.time_series.preprocessing.preprocessing_ts import StandardScalerPreprocessing
from RADAR.time_series.time_series_datasets_uci import global_load as load_time_series
from RADAR.time_series.time_series_utils import TimeSeriesProcessor
import RADAR.metrics_module as metrics_module

metrics_module = importlib.reload(metrics_module)

2026-03-12 20:42:53.350078: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 20:42:53.363586: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773344573.379840  207793 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773344573.384675  207793 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-12 20:42:53.402708: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

## Build the UCI Time-Series Benchmark

The two datasets are prepared with the same preprocessing ideas used in `test_transformers.ipynb` and `test_tsfedl.ipynb`.

- `ai4i_2020_predictive_maintenance_dataset`: anomaly labels come directly from `Machine failure`.
- `metro_interstate_traffic_volume`: anomaly labels are derived from extreme traffic volume values using the 5th and 95th percentiles.

For the time-series split, this notebook keeps chronological order instead of shuffling, so the last part of each series is reserved for testing.

In [4]:
WINDOW_SIZE = 24
STEP_SIZE = 1
TEST_SIZE = 0.2
METRO_LOW_Q = 0.05
METRO_HIGH_Q = 0.95

def chronological_split(X, y, test_size=0.2):
    split_idx = int(len(X) * (1 - test_size))
    return X[:split_idx], X[split_idx:], y[:split_idx], y[split_idx:]

def aggregate_window_labels(y_windows):
    y_windows = np.asarray(y_windows)
    if y_windows.ndim == 1:
        return y_windows.astype(int)
    return (y_windows.sum(axis=1) > 0).astype(int)

def prepare_ai4i_dataset(window_size=WINDOW_SIZE, step_size=STEP_SIZE, test_size=TEST_SIZE):
    X, y = load_time_series('ai4i_2020_predictive_maintenance_dataset')
    labels = y['Machine failure'].astype(int).to_numpy()
    X = X.drop(columns=['Type'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)

    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=False)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows = processor.process_train_test(X_train, y_train, X_test, y_test)

    return {
        'dataset': 'ai4i_2020_predictive_maintenance_dataset',
        'X_train_windows': np.asarray(X_train_windows, dtype=np.float32),
        'X_test_windows': np.asarray(X_test_windows, dtype=np.float32),
        'y_test_windows': np.asarray(y_test_windows),
        'y_test_labels': aggregate_window_labels(y_test_windows),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'window_size': window_size,
        'train_windows': len(X_train_windows),
        'test_windows': len(X_test_windows),
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'positive_ratio_windows': round(float(np.mean(aggregate_window_labels(y_test_windows))), 4),
        'label_note': 'Machine failure from UCI target',
    }

def prepare_metro_dataset(window_size=WINDOW_SIZE, step_size=STEP_SIZE, test_size=TEST_SIZE, low_q=METRO_LOW_Q, high_q=METRO_HIGH_Q):
    X, y = load_time_series('metro_interstate_traffic_volume')
    traffic_volume = y['traffic_volume'].astype(float)
    low_threshold = float(traffic_volume.quantile(low_q))
    high_threshold = float(traffic_volume.quantile(high_q))
    labels = ((traffic_volume <= low_threshold) | (traffic_volume >= high_threshold)).astype(int).to_numpy()

    X = X.drop(columns=['date_time', 'holiday', 'weather_main', 'weather_description'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)

    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=False)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows = processor.process_train_test(X_train, y_train, X_test, y_test)

    return {
        'dataset': 'metro_interstate_traffic_volume',
        'X_train_windows': np.asarray(X_train_windows, dtype=np.float32),
        'X_test_windows': np.asarray(X_test_windows, dtype=np.float32),
        'y_test_windows': np.asarray(y_test_windows),
        'y_test_labels': aggregate_window_labels(y_test_windows),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'window_size': window_size,
        'train_windows': len(X_train_windows),
        'test_windows': len(X_test_windows),
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'positive_ratio_windows': round(float(np.mean(aggregate_window_labels(y_test_windows))), 4),
        'label_note': f'Extreme traffic volume: <= q{low_q:.2f} or >= q{high_q:.2f}',
        'low_threshold': round(low_threshold, 3),
        'high_threshold': round(high_threshold, 3),
    }

dataset_configs = {
    'ai4i': prepare_ai4i_dataset(),
    'metro_interstate': prepare_metro_dataset(),
}

Metadata: {'uci_id': 601, 'name': 'AI4I 2020 Predictive Maintenance Dataset', 'repository_url': 'https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/601/data.csv', 'abstract': 'The AI4I 2020 Predictive Maintenance Dataset is a synthetic dataset that reflects real predictive maintenance data encountered in industry.', 'area': 'Computer Science', 'tasks': ['Classification', 'Regression', 'Causal-Discovery'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 10000, 'num_features': 6, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], 'index_col': ['UID', 'Product ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Wed Feb 14 2024', 'dataset_doi': '10.24432/C5HS5C', 'creators': [], 'intro_paper': {'ID': 386, 'type': 'NATIVE', 'title': 'Explainable Artificial 

In [5]:
dataset_summary = pd.DataFrame([
    {
        'dataset_key': dataset_key,
        'dataset_name': config['dataset'],
        'samples': config['n_samples'],
        'features': config['n_features'],
        'window_size': config['window_size'],
        'train_windows': config['train_windows'],
        'test_windows': config['test_windows'],
        'positive_ratio_points': config['positive_ratio_points'],
        'positive_ratio_windows': config['positive_ratio_windows'],
        'label_note': config['label_note'],
    }
    for dataset_key, config in dataset_configs.items()
]).reset_index(drop=True)

display(dataset_summary)

,dataset_key,dataset_name,samples,features,window_size,train_windows,test_windows,positive_ratio_points,positive_ratio_windows,label_note
0,ai4i,ai4i_2020_predictive_maintenance_dataset,10000,5,24,7977,1977,0.0339,0.3242,Machine failure from UCI target
1,metro_interstate,metro_interstate_traffic_volume,48204,4,24,38540,9618,0.1006,0.7486,Extreme traffic volume: <= q0.05 or >= q0.95


## Transformer Benchmark on the Two UCI Datasets

The experiment uses the three transformer-style models already demonstrated in `test_transformers.ipynb`: vanilla Transformer, Informer, and Autoformer.

For anomaly detection, the models are trained in reconstruction mode (`fit(X_train_windows)`), anomaly scores come from reconstruction error, and a binary label is predicted using the default threshold $\mu + 3\sigma$.

In [6]:
def tensor_to_numpy(values):
    if hasattr(values, 'detach'):
        values = values.detach().cpu().numpy()
    return np.asarray(values)

def score_to_labels(scores):
    scores = np.asarray(scores, dtype=float).ravel()
    threshold = scores.mean() + 3 * scores.std()
    return (scores > threshold).astype(int), float(threshold)

transformer_model_configs = [
    {
        'algorithm_': 'transformer',
        'd_model': 64,
        'd_qk': 64,
        'd_v': 64,
        'n_layers': 2,
        'n_heads': 8,
        'ulayers_feedfwd': 128,
        'dropout_rate': 0.1,
        'attns_outs': False,
        'train_epochs': 5,
        'batch_size': 32,
        'lr': 1e-3,
    },
    {
        'algorithm_': 'informer',
        'd_model': 64,
        'n_heads': 8,
        'e_layers': 2,
        'd_layers': 1,
        'd_ff': 128,
        'factor': 5,
        'dropout': 0.1,
        'attn': 'prob',
        'activation': 'gelu',
        'output_attention': False,
        'distil': True,
        'mix': True,
        'train_epochs': 5,
        'batch_size': 32,
        'lr': 1e-3,
    },
    {
        'algorithm_': 'autoformer',
        'd_model': 64,
        'n_heads': 8,
        'e_layers': 2,
        'd_layers': 1,
        'd_ff': 128,
        'factor': 5,
        'moving_avg': 5,
        'dropout': 0.1,
        'activation': 'gelu',
        'output_attention': False,
        'train_epochs': 5,
        'batch_size': 32,
        'lr': 1e-3,
    },
]

transformer_results = []

for dataset_key, config in dataset_configs.items():
    input_dim = config['X_train_windows'].shape[2]
    seq_len = config['window_size']
    y_true = config['y_test_labels']

    print(f'\nDataset: {config["dataset"]}')
    print(f'Features: {input_dim} | Train windows: {config["train_windows"]} | Test windows: {config["test_windows"]}')

    for model_template in transformer_model_configs:
        model_params = dict(model_template)
        algorithm_name = model_params['algorithm_']

        if algorithm_name == 'transformer':
            model_params.update({
                'label_parser': None,
                'size_enc_in': input_dim,
                'size_dec_in': input_dim,
                'seq_len': seq_len,
            })
        elif algorithm_name == 'informer':
            model_params.update({
                'label_parser': None,
                'enc_in': input_dim,
                'dec_in': input_dim,
                'c_out': input_dim,
                'seq_len': seq_len,
                'label_len': seq_len,
                'out_len': seq_len,
            })
        elif algorithm_name == 'autoformer':
            model_params.update({
                'label_parser': None,
                'enc_in': input_dim,
                'dec_in': input_dim,
                'c_out': input_dim,
                'seq_len': seq_len,
                'label_len': seq_len,
                'pred_len': seq_len,
            })

        model = transformers.TransformersAnomalyDetection(**model_params)

        train_start = time.time()
        model.fit(config['X_train_windows'])
        train_time = time.time() - train_start

        inference_start = time.time()
        scores = tensor_to_numpy(model.decision_function(config['X_test_windows'])).ravel()
        y_pred, threshold = score_to_labels(scores)
        inference_time = time.time() - inference_start

        finite_scores = np.isfinite(scores).all()
        accuracy = metrics_module.metric_accuracy(y_true, y_pred) / 100
        precision = metrics_module.metric_precision(y_true, y_pred)
        recall = metrics_module.metric_recall(y_true, y_pred)
        f1 = metrics_module.metric_F1score(y_true, y_pred)
        roc_auc = metrics_module.metric_AUC_ROC_scores(y_true, scores) if finite_scores else np.nan
        pr_auc = metrics_module.metric_PR_AUC(y_true, scores) if finite_scores else np.nan

        print(f'  Model: {algorithm_name}')
        print(f'    Accuracy={accuracy:.3f} | Precision={precision:.3f} | Recall={recall:.3f} | F1={f1:.3f}')
        if finite_scores:
            print(f'    ROC AUC={roc_auc:.3f} | PR AUC={pr_auc:.3f} | threshold={threshold:.4f}')

        transformer_results.append({
            'dataset_key': dataset_key,
            'dataset_name': config['dataset'],
            'algorithm': algorithm_name,
            'window_size': seq_len,
            'n_features': input_dim,
            'train_windows': config['train_windows'],
            'test_windows': config['test_windows'],
            'train_time_s': round(train_time, 4),
            'inference_time_s': round(inference_time, 4),
            'threshold': round(threshold, 6),
            'accuracy': round(accuracy, 4),
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'f1': round(f1, 4),
            'roc_auc_scores': round(float(roc_auc), 4) if np.isfinite(roc_auc) else np.nan,
            'pr_auc_scores': round(float(pr_auc), 4) if np.isfinite(pr_auc) else np.nan,
        })

transformer_results_df = pd.DataFrame(transformer_results).sort_values(
    ['dataset_name', 'pr_auc_scores', 'roc_auc_scores', 'f1'],
    ascending=[True, False, False, False],
    na_position='last',
).reset_index(drop=True)

display(transformer_results_df)


Dataset: ai4i_2020_predictive_maintenance_dataset
Features: 5 | Train windows: 7977 | Test windows: 1977
Train Params: {'train_epochs': 5, 'batch_size': 32, 'lr': 0.001, 'label_parser': None} 
Model Params: {'algorithm_': 'transformer', 'd_model': 64, 'd_qk': 64, 'd_v': 64, 'n_layers': 2, 'n_heads': 8, 'ulayers_feedfwd': 128, 'dropout_rate': 0.1, 'attns_outs': False, 'size_enc_in': 5, 'size_dec_in': 5, 'seq_len': 24}


/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([32, 24, 5])) that is different to the input size (torch.Size([24, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([9, 24, 5])) that is different to the input size (torch.Size([24, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1/5, Loss: 0.709522
Epoch 2/5, Loss: 0.735772
Epoch 3/5, Loss: 0.728336
Epoch 4/5, Loss: 0.712873
Epoch 5/5, Loss: 0.726011
  Model: transformer
    Accuracy=0.676 | Precision=0.000 | Recall=0.000 | F1=0.000
    ROC AUC=0.579 | PR AUC=0.486 | threshold=1.5226
Train Params: {'train_epochs': 5, 'batch_size': 32, 'lr': 0.001, 'label_parser': None} 
Model Params: {'algorithm_': 'informer', 'd_model': 64, 'n_heads': 8, 'e_layers': 2, 'd_layers': 1, 'd_ff': 128, 'factor': 5, 'dropout': 0.1, 'attn': 'prob', 'activation': 'gelu', 'output_attention': False, 'distil': True, 'mix': True, 'enc_in': 5, 'dec_in': 5, 'c_out': 5, 'seq_len': 24, 'label_len': 24, 'out_len': 24}


/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([32, 24, 5])) that is different to the input size (torch.Size([24, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([9, 24, 5])) that is different to the input size (torch.Size([24, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1/5, Loss: 0.681310
Epoch 2/5, Loss: 0.616157
Epoch 3/5, Loss: 0.604170
Epoch 4/5, Loss: 0.597877
Epoch 5/5, Loss: 0.596674
  Model: informer
    Accuracy=0.677 | Precision=1.000 | Recall=0.003 | F1=0.006
    ROC AUC=0.718 | PR AUC=0.576 | threshold=1.0061
Train Params: {'train_epochs': 5, 'batch_size': 32, 'lr': 0.001, 'label_parser': None} 
Model Params: {'algorithm_': 'autoformer', 'd_model': 64, 'n_heads': 8, 'e_layers': 2, 'd_layers': 1, 'd_ff': 128, 'factor': 5, 'moving_avg': 5, 'dropout': 0.1, 'activation': 'gelu', 'output_attention': False, 'enc_in': 5, 'dec_in': 5, 'c_out': 5, 'seq_len': 24, 'label_len': 24, 'pred_len': 24}


...

/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([32, 24, 4])) that is different to the input size (torch.Size([24, 4])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([12, 24, 4])) that is different to the input size (torch.Size([24, 4])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1/5, Loss: 0.854283
Epoch 2/5, Loss: 0.825574
Epoch 3/5, Loss: 0.828484
Epoch 4/5, Loss: 0.826400
Epoch 5/5, Loss: 0.828026
  Model: informer
    Accuracy=0.251 | Precision=0.000 | Recall=0.000 | F1=0.000
    ROC AUC=0.462 | PR AUC=0.708 | threshold=0.6341
Train Params: {'train_epochs': 5, 'batch_size': 32, 'lr': 0.001, 'label_parser': None} 
Model Params: {'algorithm_': 'autoformer', 'd_model': 64, 'n_heads': 8, 'e_layers': 2, 'd_layers': 1, 'd_ff': 128, 'factor': 5, 'moving_avg': 5, 'dropout': 0.1, 'activation': 'gelu', 'output_attention': False, 'enc_in': 4, 'dec_in': 4, 'c_out': 4, 'seq_len': 24, 'label_len': 24, 'pred_len': 24}


/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([32, 24, 4])) that is different to the input size (torch.Size([24, 4])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/data/Beatriz/Doctorado GR/RADAR_Plataform/RADAR/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([12, 24, 4])) that is different to the input size (torch.Size([24, 4])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1/5, Loss: 0.902049
Epoch 2/5, Loss: 0.888794
Epoch 3/5, Loss: 0.886361
Epoch 4/5, Loss: 0.884292
Epoch 5/5, Loss: 0.883475
  Model: autoformer
    Accuracy=0.251 | Precision=0.000 | Recall=0.000 | F1=0.000
    ROC AUC=0.552 | PR AUC=0.788 | threshold=0.4718


,dataset_key,dataset_name,algorithm,window_size,n_features,train_windows,test_windows,train_time_s,inference_time_s,threshold,accuracy,precision,recall,f1,roc_auc_scores,pr_auc_scores
0,ai4i,ai4i_2020_predictive_maintenance_dataset,autoformer,24,5,7977,1977,86.3769,14.6512,0.697205,0.6834,1.0000,0.0234,0.0457,0.7374,0.6216
1,ai4i,ai4i_2020_predictive_maintenance_dataset,informer,24,5,7977,1977,41.4636,6.7950,1.006095,0.6768,1.0000,0.0031,0.0062,0.7181,0.5763
2,ai4i,ai4i_2020_predictive_maintenance_dataset,transformer,24,5,7977,1977,76.1314,5.1515,1.522584,0.6758,0.0000,0.0000,0.0000,0.5790,0.4861
3,metro_interstate,metro_interstate_traffic_volume,autoformer,24,4,38540,9618,392.2974,64.8212,0.471836,0.2514,0.0000,0.0000,0.0000,0.5522,0.7880
4,metro_interstate,metro_interstate_traffic_volume,transformer,24,4,38540,9618,345.0796,23.3314,0.902914,0.2555,0.6489,0.0118,0.0232,0.4644,0.7394
5,metro_interstate,metro_interstate_traffic_volume,informer,24,4,38540,9618,194.3887,28.9757,0.634092,0.2513,0.0000,0.0000,0.0000,0.4622,0.7081


## Per-Dataset Summary

This compact summary highlights the best score-based and label-based result obtained by each architecture on each dataset.

In [7]:
transformer_summary_df = (
    transformer_results_df.groupby(['dataset_name', 'algorithm'], as_index=False)
    .agg({
        'accuracy': 'max',
        'precision': 'max',
        'recall': 'max',
        'f1': 'max',
        'roc_auc_scores': 'max',
        'pr_auc_scores': 'max',
        'train_time_s': 'mean',
        'inference_time_s': 'mean',
    })
    .sort_values(['dataset_name', 'pr_auc_scores', 'roc_auc_scores'], ascending=[True, False, False])
    .reset_index(drop=True)
)

display(transformer_summary_df)

,dataset_name,algorithm,accuracy,precision,recall,f1,roc_auc_scores,pr_auc_scores,train_time_s,inference_time_s
0,ai4i_2020_predictive_maintenance_dataset,autoformer,0.6834,1.0000,0.0234,0.0457,0.7374,0.6216,86.3769,14.6512
1,ai4i_2020_predictive_maintenance_dataset,informer,0.6768,1.0000,0.0031,0.0062,0.7181,0.5763,41.4636,6.7950
2,ai4i_2020_predictive_maintenance_dataset,transformer,0.6758,0.0000,0.0000,0.0000,0.5790,0.4861,76.1314,5.1515
3,metro_interstate_traffic_volume,autoformer,0.2514,0.0000,0.0000,0.0000,0.5522,0.7880,392.2974,64.8212
4,metro_interstate_traffic_volume,transformer,0.2555,0.6489,0.0118,0.0232,0.4644,0.7394,345.0796,23.3314
5,metro_interstate_traffic_volume,informer,0.2513,0.0000,0.0000,0.0000,0.4622,0.7081,194.3887,28.9757


In [ ]:
results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

results_main_path = results_dir / 'uci_transformers_results.csv'
results_summary_path = results_dir / 'uci_transformers_summary.csv'

transformer_results_df.to_csv(results_main_path, index=False)
transformer_summary_df.to_csv(results_summary_path, index=False)

print(f'Saved detailed results to: {results_main_path}')
print(f'Saved summary results to: {results_summary_path}')